# पाठ १७ - Foundry Local र Qwen सँग स्थानीय AI एजेन्टहरू सिर्जना गर्दै

यस नोटबुकमा तपाईंले एउटा **स्थानीय इन्जिनियरिङ सहायक** निर्माण गर्नुहुन्छ जुन पूरै तपाईंको कार्यस्थलमा चल्छ। कुनै पनि बिन्दुमा क्लाउड इन्फ्रेन्स प्रयोग हुँदैन। सहायकले गर्न सक्छ:

१. **उपकरणहरू कल गर्ने** Qwen फंक्शन कलिङ मार्फत Foundry Local बाट।
२. **फाइलहरू सूचीबद्ध र पढ्ने** स्यान्डबक्स गरिएको प्रोजेक्ट डाइरेक्टरी भित्र।
३. **कोड विश्लेषण गर्ने** सरल मेट्रिक्सहरूसँग।
४. **डकुमेण्टेशन खोज्ने** स्थानीय RAG (Chroma) को साथ।
५. **स्थानीय MCP सर्भर प्रयोग गर्ने** (यदि कुनै कन्फिगर गरिएको छैन भने दयालु रूपमा छोडिन्छ)।

एजेन्ट कोड क्लाउड पाठहरूसँग लगभग उस्तै देखिन्छ — केवल क्लाइन्ट इन्डपोइन्ट क्लाउडबाट `localhost` मा सरेको छ।


## सेटअप

यो नोटबुक चलाउनु अघि:

1. **Microsoft Foundry Local स्थापना गर्नुहोस्** (तपाईंको OS को लागि [कागजात](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) हेर्नुहोस्)।
2. **Qwen मोडेल डाउनलोड र सुरु गर्नुहोस्:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. तलका Python प्याकेजहरू स्थापना गर्नुहोस्।

सबै कुरा स्थानीय रूपमा चल्छ। लगभग ८ GB RAM भएको मेसिन यथार्थपरक न्यूनतम हो।


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. फाउन्ड्री स्थानीयसँग जडान गर्नुहोस्

`FoundryLocalManager` आवश्यक परे मोडेल डाउनलोड गर्छ, स्थानीय सेवा सुरु गर्छ, र हामीलाई **OpenAI-संग संगत एन्डपोइन्ट** दिन्छ। त्यसपछि हामी स्ट्यान्डर्ड OpenAI SDK लाई यसतर्फ संकेत गर्छौं। API कुञ्जी एउटा स्थानीय प्लेसहोल्डर हो — कुनै क्लाउड प्रमाणपत्र समावेश छैन।


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. स्थानीय उपकरणहरू (स्यान्डबक्स गरिएको फाइल अपरेसनहरू)

हामी डिस्कमा सानो नमुना परियोजना सिर्जना गर्छौं, त्यसपछि त्यो परियोजना रुटमा सीमित उपकरणहरू परिभाषित गर्छौं। स्यान्डबक्स जाँच स्थानीय रूपमा पनि महत्त्वपूर्ण छ: एक उपकरण जुन मनमाने पथहरू पढ्छ तपाईंको प्रयोगकर्ताको अनुमति अनुसार चल्छ र तपाईंले पहुँच गर्न सक्ने कुनै पनि कुरा स्पर्श गर्न सक्छ।


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. क्रोमा संग स्थानीय RAG

हामीले केही साना कागजातका अंशहरूलाई स्थानीय क्रोमा संग्रहमा समाहित गर्छौं। क्रोमा इन-प्रोसेस सञ्चालन हुन्छ र डिस्कमा भेक्टरहरू सङ्ग्रह गर्छ — कुनै सर्भर, कुनै क्लाउड छैन। `search_docs` उपकरणले सोधिएको प्रश्नका लागि सबैभन्दा सान्दर्भिक अंशहरू फेला पार्छ।


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. उपकरण-कलिङ्ग लूप

अब हामी OpenAI उपकरण स्कीमाको प्रयोग गरेर उपकरणहरू मोडेलसँग दर्ता गर्छौं र स्ट्यान्डर्ड उपकरण-कलिङ्ग लूप चलाउँछौं — मोडेलले एउटा उपकरणको अनुरोध गर्छ, हामीले त्यसलाई स्थानीय रूपमा कार्यान्वयन गर्छौं, परिणाम फिर्ता दिन्छौं, र दोहोर्याउँछौं जबसम्म मोडेलले अन्तिम उत्तर उत्पादन गर्दैन। Qwen को भरपर्दो फंक्शन कलिङ्गले यो डिभाइसमा काम गर्न सक्ने बनाउँछ।


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## ५. स्थानीय MCP (वैकल्पिक)

MCP एक ट्रान्सपोर्ट हो, क्लाउड सेवा होइन — स्थानीय प्रक्रिया रूपमा `stdio` माथि MCP सर्भर चलाउन सकिन्छ। तलको सेलले देखाउँछ कि तपाईंसँग कन्फिगर गरिएको स्थानीय MCP सर्भर (जस्तै फाइल प्रणाली सर्भर) भएमा कसरी जडान गर्ने। यदि `LOCAL_MCP_COMMAND` सेट गरिएको छैन भने यो सहजै स्किप गर्छ, त्यसैले नोटबुक अझै पनि अन्तदेखि अन्तसम्म चल्छ।

सुरक्षा नोट: स्थानीय MCP सर्भर तपाइँको प्रयोगकर्ता अनुमति साथ चल्छ। यसलाई प्रोजेक्ट डाइरेक्टरीमा सिमित गर्नुहोस् र त्यसका नतिजाहरूमा काम गर्नु अघि तिनीहरूलाई प्रमाणीकरण गर्नुहोस्।


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## सारांश

तपाईंले यस्तो ईन्जिनियरिङ सहायक निर्माण गर्नुभयो जुन पूर्ण रूपमा तपाईंको कम्प्युटरमा चल्दछ:

- **Foundry Local** ले OpenAI-संग मेल खाने अन्तर्क्रियाअन्तर्गत **Qwen** मोडल सेवा दियो — त्यसैले एजेन्ट कोड क्लाउड पाठहरू संग मेल खान्छ।
- **Sandboxed tools** ले एजेन्टलाई प्रोजेक्ट डाइरेक्टरी छोड्न नदिइ फाइल पहुँच र कोड विश्लेषण प्रदान गर्‍यो।
- **Chroma** ले दस्तावेजीकरणमा **स्थानीय RAG** उपलब्ध गरायो।
- **Local MCP** ले देखायो कसरी MCP इकोसिस्टमलाई अफलाइन पुन: प्रयोग गर्ने।

कुनै पनि बेला क्लाउड इन्फरेन्स प्रयोग गरिएको थिएन।

### चुनौती

स्थानीय एजेन्टलाई विस्तार गर्नुहोस्:

1. **धेरै MCP सर्भरहरूसँग काम गर्ने** — फाइलसिस्टम सर्भर र गिट सर्भरसँग जडान गर्नुहोस् र एजेन्टलाई तीमध्ये रोज्न दिनुहोस्।
2. **स्थानीय मेमोरी प्रयोग गर्ने** — सहायकले नोटबुक पुनः सुरू गर्दा पहिलेका कुराकानी सम्झन छोटो इतिहास डिस्कमा सुरक्षित गर्ने।
3. **स्थानीय बहु-एजेन्ट समन्वय समर्थन गर्ने** — दोस्रो स्थानीय एजेन्ट (जस्तै समीक्षक) थप्नुहोस् र उनीहरूलाई मिलेर कार्यमा सहकार्य गर्न दिनुहोस्।

अर्को पाठमा तपाईं सिक्नुहुनेछ कसरी तैनाथ गरिएको AI एजेन्टहरूलाई सुरक्षीत गर्ने। 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
